# Judge step (one grid run)

Re-opens an already-tagged TruLens run of `V1_FIRST_PASS` and computes the judge's metrics: built-in `correctness`
plus our `tag_groundedness` and `tag_coherence` (from `v1_trulens.py`). Runs headless as a Code Bundle on a compute
pool, so TruLens 2.16 and its libraries come from `requirements.txt`, the same versions as the laptop runs.

Arguments: `sys.argv[1]` = setup (e.g. `C01`), `sys.argv[2]` = repeat (default `1`), `sys.argv[3]` = TruLens run
suffix (default `_c`).

In [ ]:
import json
import os
import re
import sys
import time

os.environ.setdefault("TRULENS_OTEL_TRACING", "1")

from snowflake.snowpark.context import get_active_session
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector

from v1_trulens import DB, V1FirstPass, tag_metrics

args = sys.argv[1:] + [None] * 3
config_id = args[0] or "C01"
repeat = int(args[1] or 1)
suffix = args[2] or "_c"
if not re.fullmatch(r"C0[1-8]", config_id) or not re.fullmatch(r"_[a-z]", suffix):
    raise ValueError(f"unexpected arguments: {sys.argv[1:]}")

session = get_active_session()
session.use_database("VOC_EVAL")
session.use_schema("HARNESS")
print("setup", config_id, "repeat", repeat, "suffix", suffix)

In [ ]:
# Re-attach to the registered app version and the existing run (no model calls here, only the judge).
judge = session.sql(f"SELECT MODEL FROM {DB}.V1_MODEL_CONFIG WHERE ROLE = 'judge'").collect()[0][0]
version = session.sql(f"SELECT APP_VERSION FROM {DB}.V1_MODEL_CONFIG WHERE CONFIG_ID = ?",
                      params=[config_id]).collect()[0][0]
results_id = f"{config_id}_SAMPLE_r{repeat}_trulens"
run_name = results_id + suffix

app = V1FirstPass(session, config_id, "SAMPLE", repeat, results_id)
session.sql(f"DROP TABLE IF EXISTS {app.stage}").collect()
tru_app = TruApp(app, app_name="V1_FIRST_PASS", app_version=version,
                 connector=SnowflakeConnector(snowpark_session=session), main_method=app.tag_comment)
run = tru_app.get_run(run_name)
print("run", run_name, "version", version, "judge", judge)

In [ ]:
# The judge step. Scores appear in Snowsight > AI & ML > Evaluations > V1_FIRST_PASS.
t0 = time.time()
status = run.compute_metrics(metrics=["correctness"] + tag_metrics(session, judge))
print(json.dumps({"run": run_name, "status": str(status), "seconds": round(time.time() - t0)}))